# Testing del refactor output

In [1]:
import os
import sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.append(project_root)

## Recuperacion de mensajes y Deserealizacion

In [2]:
from app.services.dynamodb_queries import get_latests_messages, deserialize_item
from app.models.ChatMessage import ChatHistoryElement, ChatHistoryResponse

In [22]:
#lookup key
user_id = "refactor-output-test-2"
conv_id = "refactor-output-test-2"
primary_key = f"USER#{user_id}#CONV#{conv_id}"

In [23]:
raw_messages = get_latests_messages(primary_key=primary_key, limit=10)

In [25]:
raw_messages['Items']

[{'content': {'M': {'text': {'S': '¡Hola! Para ayudarte mejor, ¿podrías decirme en qué **ubicación** buscas la propiedad? ¿Se trata de una **propiedad** tipo casa, departamento, etc.? ¿Estamos buscando una **transacción** de compra o alquiler? ¡Esto nos ayudará a enfocar la búsqueda!'}}},
  'metadata': {'M': {'additionalProp1': {'M': {}},
    'stage': {'S': 'extract'},
    'lead': {'M': {'metraje_minimo': {'NULL': True},
      'numero_dormitorios': {'NULL': True},
      'ubicacion': {'NULL': True},
      'tipo_propiedad': {'NULL': True},
      'presupuesto': {'NULL': True},
      'transaccion': {'NULL': True},
      'pet_friendly': {'NULL': True},
      'amenidades': {'NULL': True},
      'numero_banos': {'NULL': True},
      'cercania': {'NULL': True}}}}},
  'SK': {'S': 'TIMESTAMP#2025-07-16T07:38:57.387815Z'},
  'PK': {'S': 'USER#refactor-output-test-2#CONV#refactor-output-test-2'},
  'role': {'S': 'assistant'},
  'content_type': {'S': 'text'}},
 {'content': {'M': {'text': {'S': 'Hol

In [24]:
message_list = []
verbose = True
for raw_item in raw_messages.get('Items'):
    item = deserialize_item(raw_item)
    if verbose == False:
        item.pop('SK')
        item.pop('metadata')
    message_list.append(ChatHistoryElement(**item))


In [21]:
ChatHistoryResponse(history=message_list)

ChatHistoryResponse(history=[ChatHistoryElement(role='assistant', content={'properties': [{'score': Decimal('0.0062553654'), 'id': '0d641720-f52f-49bd-9b90-6fc964380d09', 'text': 'Alquiler: Apartamento moderno en Urbanización Cooviecma, Santiago de Surco – Apartamento confortable y acogedor ubicado en el corazón de la urbanización Cooviecma, con acceso fácil a los principales puntos del distrito de Santiago de Surco. – local – alquiler – Batalla de Miraflores, Urbanización Cooviecma, Santiago de Surco, Lima, Lima Metropolitana, Lima, 15049, Perú – available'}, {'score': Decimal('0.0059607797'), 'id': '4cb7f6c1-8e26-4f09-a076-4d668f85895e', 'text': 'Casa alquiler en Urbanización Cooviecma, Santiago de Surco – Casa disponible para alquilar en la tranquila Urbanización Cooviecma, en Santiago de Surco. Área comoda con instalaciones modernas y servicios completos. – casa – alquiler – Calle Teniente José Melitón Rodríguez, Urbanización Cooviecma, Santiago de Surco, Lima, Lima Metropolitana, 

## Escritura de Mensajes y Serializacion

In [2]:
from app.core.config import DYNAMODB_TABLE
from app.services.dynamodb_queries import write_message, serialize_item, message_wrapper_flex
from app.models.ChatMessage import ChatMessage

In [3]:
#lookup key
user_id = "test-chat-1"
conv_id = "test-chat-1"
primary_key = f"USER#{user_id}#CONV#{conv_id}"

### Dato Texto

In [4]:
data_dict = {
    "PK" : primary_key,
    "role" : "user",
    "content_type": "text",
    "content": {
        "text" : "test text"
    },
    "metadata": {
        'stage': 'recommend'
    }
}

In [5]:
message_dict = message_wrapper_flex(data_dict)

In [6]:
message_dict

{'PK': 'USER#test-chat-1#CONV#test-chat-1',
 'role': 'user',
 'content_type': 'text',
 'content': {'text': 'test text'},
 'metadata': {'stage': 'recommend'},
 'SK': 'TIMESTAMP#2025-07-16T01:32:47.554961Z'}

In [7]:
formatted_message = ChatMessage(**message_dict)

In [8]:
print(formatted_message)

PK='USER#test-chat-1#CONV#test-chat-1' SK='TIMESTAMP#2025-07-16T01:32:47.554961Z' role='user' content_type='text' content={'text': 'test text'} metadata={'stage': 'recommend'}


In [9]:
ser_item = serialize_item(formatted_message)
ser_item

{'PK': {'S': 'USER#test-chat-1#CONV#test-chat-1'},
 'SK': {'S': 'TIMESTAMP#2025-07-16T01:32:47.554961Z'},
 'role': {'S': 'user'},
 'content_type': {'S': 'text'},
 'content': {'M': {'text': {'S': 'test text'}}},
 'metadata': {'M': {'stage': {'S': 'recommend'}}}}

In [10]:
write_message(DYNAMODB_TABLE, ser_item)

### Dato Propiedades

In [12]:
recommendation_list = [
    {'id': '4c165b51-8556-4a11-8294-62b6a34d09b3'},
    {'id': '0d75ad71-7bf6-4086-9c9e-2f6af3f381e8'},
    {'id': 'a7c9d240-355b-490c-8ead-84bef127199f'}
]

In [13]:
data_dict = {
    "PK" : primary_key,
    "role" : "user",
    "content_type": "property_list",
    "content": {
        "properties" : recommendation_list
    },
    "metadata": {
        'stage': 'recommend'
    }
}

In [17]:
message_dict = message_wrapper_flex(data_dict)
message_dict

{'PK': 'USER#test-chat-1#CONV#test-chat-1',
 'role': 'user',
 'content_type': 'property_list',
 'content': {'properties': [{'id': '4c165b51-8556-4a11-8294-62b6a34d09b3'},
   {'id': '0d75ad71-7bf6-4086-9c9e-2f6af3f381e8'},
   {'id': 'a7c9d240-355b-490c-8ead-84bef127199f'}]},
 'metadata': {'stage': 'recommend'},
 'SK': 'TIMESTAMP#2025-07-16T01:43:54.971634Z'}

In [22]:
formatted_message = ChatMessage(**message_dict)
formatted_message

ChatMessage(PK='USER#test-chat-1#CONV#test-chat-1', SK='TIMESTAMP#2025-07-16T01:43:54.971634Z', role='user', content_type='property_list', content={'properties': [{'id': '4c165b51-8556-4a11-8294-62b6a34d09b3'}, {'id': '0d75ad71-7bf6-4086-9c9e-2f6af3f381e8'}, {'id': 'a7c9d240-355b-490c-8ead-84bef127199f'}]}, metadata={'stage': 'recommend'})

In [23]:
ser_item = serialize_item(formatted_message)
ser_item

{'PK': {'S': 'USER#test-chat-1#CONV#test-chat-1'},
 'SK': {'S': 'TIMESTAMP#2025-07-16T01:43:54.971634Z'},
 'role': {'S': 'user'},
 'content_type': {'S': 'property_list'},
 'content': {'M': {'properties': {'L': [{'M': {'id': {'S': '4c165b51-8556-4a11-8294-62b6a34d09b3'}}},
     {'M': {'id': {'S': '0d75ad71-7bf6-4086-9c9e-2f6af3f381e8'}}},
     {'M': {'id': {'S': 'a7c9d240-355b-490c-8ead-84bef127199f'}}}]}}},
 'metadata': {'M': {'stage': {'S': 'recommend'}}}}

In [24]:
write_message(DYNAMODB_TABLE, ser_item)

## Handler stage 1

In [5]:
from app.services.stages.stage1_extract import handle
from app.services.chatbot_engine import convert_to_conversation
from app.services.dynamodb_queries import get_latests_messages

In [6]:
user_id = "refactor-output-test-1"
conv_id = "refactor-output-test-1"
primary_key = f"USER#{user_id}#CONV#{conv_id}"

In [7]:
latest_messages = get_latests_messages(primary_key, limit=10)
latest_conversation = convert_to_conversation(latest_messages)

In [8]:
latest_conversation

[{'role': 'assistant',
  'content': '¡Hola! Para ayudarte mejor, ¿podrías decirme en qué ciudad o barrio te gustaría vivir? ¿Buscas una casa, departamento u otra propiedad? ¿Es para comprar o alquilar? ¡Esto nos ayudará a encontrar lo que buscas!'},
 {'role': 'user', 'content': 'Hola'}]

In [12]:
conversation =     [{'role': 'user',
        'content': [{"text": "Hello"}]},
       {'role': 'assistant',
        'content': [{"text": "Hola, en que puedo ayudarte?"}]}, 
    ]


In [13]:
response = handle(conversation)

In [14]:
response

{'conversation': [{'role': 'user', 'content': [{'text': 'Hello'}]},
  {'role': 'assistant',
   'content': [{'text': 'Hola, en que puedo ayudarte?'}]}],
 'base_prompt': 'Simula ser un asesor inmobiliario que guía al usuario con preguntas para entender qué tipo de propiedad desea el cliente llenando los datos REQUERIDOS. Sé breve pero cordial y amigable. (máx 50 palabras).❗Actualmente los datos FALTANTES son: {datos_faltantes} <- Pregunta por estos ❗ Como contexto ten en cuenta los datos que podemos recolectar y su descripción:{data_info}',
 'lead_prompt': 'A partir del siguiente historial de mensajes de usuario, extrae únicamente los datos explícitamente mencionados.\n\n            ❗No completes campos por inferencia.  \n            ❗Si el dato no está mencionado literalmente o con sinónimos claros, déjalo como `None`.\n\n            No asumas que busca alquiler solo porque menciona "departamento", ni que busca compra porque menciona "terreno". Solo responde con datos explícitos.\n\n   

In [15]:
response.get('model_response')

'Para orientarte mejor, ¿podrías decirme en qué ubicación buscas la propiedad? Además, ¿estás buscando una casa, departamento u otra tipo de propiedad? ¿Y para qué tipo de transacción? ¡Esto nos ayudará a encontrar lo que buscas!'

### Handler Stage 2

In [4]:
from app.services.stages.stage2_recommend import handler as stage2_handler
from app.services.stages.stage2_recommend import create_lead_description
from app.models.PropertyLead import PropertyLead

In [7]:
property_lead = PropertyLead(
    ubicacion='Lima, Los Olivos',
    tipo_propiedad=['departamento'],
    transaccion='alquiler'
)

In [10]:
recomendation = stage2_handler(property_lead)
recomendation

[{'id': '4c165b51-8556-4a11-8294-62b6a34d09b3',
  'text': 'Atractivo departamento en Los Olivos, Lima – Departamento moderno y acogedor en un hermoso complejo de apartamentos ubicado en el distrito de Los Olivos, Lima. – cuarto – venta – 6 de Noviembre, Los Olivos, Lima, Lima Metropolitana, Lima, 15031, Perú – available',
  'score': 0.010889926},
 {'id': '5ec6bd2c-7d2a-4e76-9557-23a03b11f00f',
  'text': 'Atractivo apartamento en el corazón de Los Olivos – Ofrecemos un apartamento moderno y cómodo en una ubicación privilegiada del distrito Los Olivos, Lima. Aprovecha este oportunidad y vive en un espacio único. – local – alquiler – Calle 17, Los Olivos, Lima, Lima Metropolitana, Lima, 15307, Perú – available',
  'score': 0.010593942},
 {'id': 'a7c9d240-355b-490c-8ead-84bef127199f',
  'text': 'Departamento elegante en exclusivo Los Olivos, Lima – Ofrecemos un moderno departamento de lujo en el prestigioso barrio de Los Olivos, a solo 10 minutos del centro de Lima. – departamento – alquil